# Части final_df в одной тетрадке

Каждая часть пишет свою таблицу в `sandbox_ai`. Следующая часть читает предыдущую таблицу, а не список из Python. Spark в конце открывает уже готовый периметр и в озеро сам не ходит.

В DRP ничего не пишется. Боевая таблица дашборда не меняется.

Сейчас в тетрадке части, которые считаются отдельным запросом: периметр договоров, CDI, CFT, операции по договору, MCC. Терминалы, тарифы и продукты остаются следующими ячейками этого же файла: им нужна таблица амортизации и оконный ранг тарифа.


In [ ]:
import calendar
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

REPORT_MONTH = "2026-07"
year_i, month_i = (int(part) for part in REPORT_MONTH.split("-"))
month_start = f"{REPORT_MONTH}-01"
month_end = f"{REPORT_MONTH}-{calendar.monthrange(year_i, month_i)[1]:02d}"
print("month", REPORT_MONTH, month_start, month_end)

In [ ]:
import getpass
import os

from rail_connectors.connection import connect

keytab_path = "/home/jovyan/test_requests/tech.keytab"
os.makedirs(os.path.dirname(keytab_path), exist_ok=True)

LAKE_USER = input("Логин Impala: ").strip()
LAKE_PASSWORD = getpass.getpass("Пароль Impala: ")
if not LAKE_USER or not LAKE_PASSWORD:
    raise RuntimeError("Логин и пароль нужны оба")

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": keytab_path,
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": LAKE_USER, "password": LAKE_PASSWORD},
)
imp._init_connection()
with imp:
    imp.execute("SELECT 1")
print("Impala ok")

In [ ]:
from IPython.display import display

def save_part(table, select_sql, mem_limit="8g"):
    imp._init_connection()
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        imp.execute(f"DROP TABLE IF EXISTS {table}")
        imp.execute(f"CREATE TABLE {table} STORED AS PARQUET AS\n{select_sql}")
        cnt = imp.fetch(f"SELECT count(*) AS n FROM {table}")
        preview = imp.fetch(f"SELECT * FROM {table} LIMIT 20")
    print(table)
    display(cnt)
    display(preview)
    if cnt is None or len(cnt) == 0 or int(cnt.iloc[0]["n"]) == 0:
        raise RuntimeError(f"Пустая таблица {table}")
    return cnt

## 01. Периметр договоров SA

Договоры эквайринга, действующие в июле, с терминалом типа P.


In [ ]:
t01 = "sandbox_ai.tmp_shestopalov_acq_01_sa_perimeter"
sql01 = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""
save_part(t01, sql01)

## Spark читает периметр

Локально, без YARN. Озеро не сканирует: открывает файлы таблицы 01.


In [ ]:
import os
from pyspark.sql import SparkSession

log4j_path = "/tmp/acq_spark_log4j2.properties"
with open(log4j_path, "w", encoding="utf-8") as log4j_file:
    log4j_file.write("\n".join([
        "status = error",
        "name = AcqSpark",
        "appender.console.type = Console",
        "appender.console.name = console",
        "appender.console.target = SYSTEM_ERR",
        "appender.console.layout.type = PatternLayout",
        "appender.console.layout.pattern = %p %c{1}: %m%n",
        "rootLogger.level = error",
        "rootLogger.appenderRef.stdout.ref = console",
    ]))
os.environ["SPARK_SUBMIT_OPTS"] = f"-Dlog4j2.configurationFile=file:{log4j_path}"

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("acq-final-parts-local")
    .master("local[*]")
    .config("spark.log.level", "ERROR")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

imp._init_connection()
with imp:
    desc = imp.fetch(f"DESCRIBE FORMATTED {t01}")
name_col, type_col = desc.columns[0], desc.columns[1]
mask = desc[name_col].astype(str).str.contains("Location", case=False, na=False)
location = str(desc.loc[mask, type_col].iloc[0]).strip()
print("location", location)
part = spark.read.parquet(location)
part.show(20, truncate=False)
print("rows ok", part.count())

## 02. CDI по ИНН из периметра


In [ ]:
t02 = "sandbox_ai.tmp_shestopalov_acq_02_cdi_map"
sql02 = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in (
      select inn from {t01}
    )
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""
save_part(t02, sql02)

## 03. CFT по CDI из предыдущей таблицы


In [ ]:
t03 = "sandbox_ai.tmp_shestopalov_acq_03_cft_map"
sql03 = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in (
    select cdi_id from {t02} where cdi_id is not null
  )
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""
save_part(t03, sql03)

## 05. Операции по договору

Это секция операций `final_df`: не по дням, а одна строка на договор. Запрос тяжёлый, может идти долго.


In [ ]:
t05 = "sandbox_ai.tmp_shestopalov_acq_05_trx_agr"
sql05 = f"""
with sa_agr as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
    and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),
fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
trx_base_raw as (
  select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{month_end}' as date), 1) as timestamp)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
),
trx_base as (
  select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
  from trx_base_raw
  group by n_trx
),
ta_raw as (
  select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
),
ta as (
  select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
  from ta_raw
  group by n_trx, n_agr
),
trx_keys as (
  select distinct n_trx
  from ta
),
trx_int_agg as (
  select cast(i.n_trx as string) as n_trx, max(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join trx_keys k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
),
tj as (
  select ta.n_agr, sa.n_cmp_client, sa.inn, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
  from ta
  join trx_base tb on tb.n_trx = ta.n_trx
  left join sa_agr sa on sa.n_agr = ta.n_agr
),
trx_agg as (
  select
    tj.n_agr,
    count(distinct tj.n_trx) as trx_cnt,
    sum(tj.n_amt_src) as trx_sum,
    sum(tj.n_amt_tax) as commission_from_ops,
    sum(coalesce(i.n_amt_fee, 0.0)) as int_component
  from tj
  left join trx_int_agg i on i.n_trx = tj.n_trx
  group by tj.n_agr
),
active_term_agg_ncmp as (
  select
    tj.n_cmp_client,
    count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
  from tj
  where tj.n_cmp_client is not null
  group by tj.n_cmp_client
),
active_term_agg_agr as (
  select
    tj.n_agr,
    count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_terms
  from tj
  where tj.n_agr is not null
  group by tj.n_agr
)
select
  t.n_agr,
  sa.inn,
  t.trx_cnt,
  t.trx_sum,
  t.commission_from_ops,
  t.int_component,
  sa.n_cmp_client,
  a.active_term_cnt,
  aa.active_terms
from trx_agg t
left join sa_agr sa on sa.n_agr = t.n_agr
left join active_term_agg_ncmp a on a.n_cmp_client = sa.n_cmp_client
left join active_term_agg_agr aa on aa.n_agr = t.n_agr
"""
save_part(t05, sql05, mem_limit="16g")

## MCC по договору

Справочник мерчантов, одна строка на договор. От месяца не зависит.


In [ ]:
t_mcc = "sandbox_ai.tmp_shestopalov_acq_mcc_by_agr"
sql_mcc = """
with sa as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client
  from ods_alpha.scd1_agreements a
  where a.acq_class = 'SA'
    and a.abs_agr_id is not null
),
raw_mcc as (
  select
    s.n_agr,
    cast(m.n_mcc as string) as mcc,
    cast(m.c_mrc_profile as string) as mcc_profile
  from sa s
  left join ods_alpha.scd1_merchants m
    on cast(m.n_cmp as string) = s.n_cmp_client
  where m.n_mcc is not null or m.c_mrc_profile is not null
)
select
  n_agr,
  group_concat(distinct mcc, ',') as mcc,
  group_concat(distinct mcc_profile, ',') as mcc_profile,
  count(distinct mcc) as mcc_cnt
from raw_mcc
group by n_agr
"""
save_part(t_mcc, sql_mcc, mem_limit="4g")

## Что остаётся в этой же тетрадке

Терминалы (04), атрибуты CFT (06), тариф (09), комиссия MPOS (10m) и продукты клиента (10b) считаются следом здесь же. Их запросы опираются на таблицу амортизации и на ранг актуального тарифа, поэтому в этот прогон они не входят.
